In [1]:
# 特征工程与数据集划分脚本
#准备
import pandas as pd
import numpy as np #numpy处理纯数值数据，这里用于把SMILES转化为Morgan指纹后，存储为numpy数组
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold #分子骨架提取,后面用来按骨架划分数据集
from rdkit import RDLogger #日志
import math
import random


In [4]:
RDLogger.DisableLog('rdApp.*') #关闭RDKit的日志输出，避免干扰

def scaffold_split(data_df, train_size=0.8, random_seed=42): #原材料、切分比例
    """
    实现 Scaffold Split（按分子骨架划分数据集）
    提取分子骨架，并按骨架包含的分子数量从大到小排序，划分训练集和测试集。
    核心目的是为了防作弊，避免数据泄露。避免训练集和测试集之间的分子结构过于相似，从而更好地评估模型的泛化能力。
    """
    # 1. 提取所有分子的骨架 (Murcko Scaffold)
    scaffolds = {}
    for idx, smiles in enumerate(data_df['std_smiles']):
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None: #防误
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)#忽略手性
            scaffolds.setdefault(scaffold, []).append(idx)
    
    # 2. 按骨架包含的分子数量从大到小排序
    scaffold_sets = [scaffold_set for _, scaffold_set in sorted(scaffolds.items(), key=lambda x: len(x[1]),  reverse=True)]
    
    # 3. 划分训练集和测试集
    train_indices = []
    test_indices = []
    train_cutoff = math.floor(train_size * len(data_df))
    '''
    划分逻辑：
    1.先按骨架包含的分子量从大到小排列，相似相同的骨架会集中在一起，打包塞进训练集，
    防止训练集和测试集之间的分子结构过于相似，导致作弊。
    2.为什么不在80%处直接切分？
    防止相同相似骨架被同时分进训练集和测试集，导致数据泄露，模型作弊。
    用for循环按骨架打包塞进训练集，直到训练集达到80%，剩下的骨架都塞进测试集。
    '''
    for scaffold_set in scaffold_sets:
        # 如果加入当前骨架后，训练集大小不超过设定的阈值，就加入训练集
        if len(train_indices) + len(scaffold_set) <= train_cutoff:
            train_indices.extend(scaffold_set)
        else:
            test_indices.extend(scaffold_set)
            # 如果因为结构问题，有一部分没分进去，补到测试集
    
    #摇匀
    random.seed(random_seed)#锁定，让每次划分结果一致
    random.shuffle(train_indices)
    random.shuffle(test_indices)
    
    return train_indices, test_indices

#特征工程
def main():
    print("="*40)
    print("开始特征工程与数据集划分...")
    print("="*40)

    # 1. 读取清洗好的数据
    df = pd.read_csv("D:/00_vscode_project/trail_work/01_my_work/s1_Lipophilicity_clean_final.csv")
    
    # 2. 提取 Morgan 指纹 (半径=2，位数=2048)
    def get_morgan_fp(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        # 返回一个 0 和 1 组成的 numpy 数组
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        return np.array(fp)
    
    print("正在提取 Morgan 指纹，这可能需要几十秒...")
    # 将指纹列表转换为二维的 numpy 矩阵 (X)
    X = np.array(df['std_smiles'].apply(get_morgan_fp).tolist())
    # 标签 (y)
    y = df['exp'].values
    
    print(f"特征矩阵 X 的形状: {X.shape}") # 应该是 (样本数, 2048)

    # 3. 进行数据集划分(8:2)
    print("正在进行 Scaffold Split...")
    train_idx, test_idx = scaffold_split(df, train_size=0.8)
    
    # 根据索引切分数据。从大矩阵里拿出一个小矩阵，训练集和测试集的特征和标签
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print(f"训练集样本数: {len(train_idx)}")
    print(f"测试集样本数: {len(test_idx)}")

    # 4. 保存划分结果（为了后续训练方便，我们将特征和标签拼接成 DataFrame 存下来）
    # 转成 DataFrame
    df_train = pd.DataFrame(X_train)
    df_train['exp'] = y_train
    df_test = pd.DataFrame(X_test)
    df_test['exp'] = y_test

    df_train.to_csv("D:/00_vscode_project/trail_work/01_my_work/s2_train_features.csv", index=False)
    df_test.to_csv("D:/00_vscode_project/trail_work/01_my_work/s2_test_features.csv", index=False)
    print("特征提取与划分完成！已保存为 train_features.csv 和 test_features.csv")

if __name__ == "__main__":
    main()

开始特征工程与数据集划分...
正在提取 Morgan 指纹，这可能需要几十秒...
特征矩阵 X 的形状: (3780, 2048)
正在进行 Scaffold Split...
训练集样本数: 3024
测试集样本数: 756
特征提取与划分完成！已保存为 train_features.csv 和 test_features.csv
